In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Load dataset inbuild iris dataset in scikit-learn
iris = load_iris()
X, y = iris.data, iris.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create and train KNN classifier
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

# Evaluate
y_pred = knn.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 1.0


## Part 2: KNN on a Real-World Weather Dataset

### Dataset Overview
We use the **Open-Meteo API** to collect hourly weather data for Berlin, Germany. The dataset includes temperature and time, and can be extended with humidity, precipitation, and wind speed.

### Objective
The goal is to build a **KNN classification model** to classify weather conditions as **Rainy or Not Rainy**.

### Steps
1. Load weather data from the API.
2. Clean and preprocess the data.
3. Select features and create target labels.
4. Train the KNN model using an ML Pipeline.
5. Evaluate accuracy and visualize results.

In [ ]:
import requests
import pandas as pd

# Open-Meteo API URL (Berlin, Germany)
url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=52.52&longitude=13.41"
    "&hourly=temperature_2m"
)

# Fetch weather data
response = requests.get(url, timeout=30)
response.raise_for_status()

data = response.json()

# Convert JSON into a DataFrame
df = pd.DataFrame(data["hourly"])

# Display dataset information
df.to_csv("weather_data.csv", index=False)  # Save to CSV for reference
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (168, 2)

Columns: ['time', 'temperature_2m']

First 5 rows:


,time,temperature_2m
0,2026-10-09T00:00,9.9
1,2026-10-09T01:00,9.7
2,2026-10-09T02:00,9.5
3,2026-10-09T03:00,9.6
4,2026-10-09T04:00,9.4


### Step 3: Data Exploration and Cleaning

We explore the weather dataset to understand its structure and quality before applying KNN.

- **Data Types:** Check the type of each column.
- **Missing Values:** Identify any missing data.
- **Duplicates:** Check for repeated records.
- **Summary Statistics:** Analyze temperature, humidity, precipitation, and wind speed.

This helps us identify any cleaning needed before training the KNN model.

In [4]:
# Check dataset information
print("Dataset Information:")
weather_df.info()

# Check missing values
print("\nMissing Values:")
print(weather_df.isnull().sum())

# Check duplicate records
print("\nDuplicate Records:")
print(weather_df.duplicated().sum())

# Display summary statistics
print("\nSummary Statistics:")
display(weather_df.describe())

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   time                  192 non-null    str    
 1   temperature_2m        192 non-null    float64
 2   relative_humidity_2m  192 non-null    int64  
 3   precipitation         192 non-null    float64
 4   wind_speed_10m        192 non-null    float64
dtypes: float64(3), int64(1), str(1)
memory usage: 7.6 KB

Missing Values:
time                    0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
dtype: int64

Duplicate Records:
0

Summary Statistics:


,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
count,192.000000,192.000000,192.000000,192.000000
mean,14.564063,77.218750,0.038542,7.310938
std,3.266490,14.468021,0.175976,4.833146
min,8.600000,40.000000,0.000000,0.400000
25%,11.975000,68.000000,0.000000,3.875000
50%,14.100000,79.000000,0.000000,6.100000
75%,16.900000,87.000000,0.000000,8.700000
max,21.500000,100.000000,1.700000,19.600000


### Step 4: Data Cleaning and Target Creation

We prepare the weather dataset by removing duplicate records and handling missing values.

We also create a target column called `rainy`:
- **1:** Rainy (precipitation > 0 mm)
- **0:** Not Rainy (precipitation = 0 mm)

This target will be used for KNN classification.

In [5]:
# Create a copy of the dataset
clean_df = weather_df.copy()

# Remove duplicate records
clean_df = clean_df.drop_duplicates()

# Remove rows with missing precipitation values
clean_df = clean_df.dropna(subset=["precipitation"])

# Create target column
clean_df["rainy"] = (
    clean_df["precipitation"] > 0
).astype(int)

# Display target distribution
print("Rainy vs Not Rainy:")
print(clean_df["rainy"].value_counts())

print("\nDataset shape:", clean_df.shape)

display(clean_df.head())

Rainy vs Not Rainy:
rainy
0    176
1     16
Name: count, dtype: int64

Dataset shape: (192, 6)


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,rainy
0,2026-10-02T00:00,14.8,68,0.0,5.6,0
1,2026-10-02T01:00,14.1,72,0.0,2.3,0
2,2026-10-02T02:00,13.7,74,0.0,3.1,0
3,2026-10-02T03:00,12.6,83,0.0,1.9,0
4,2026-10-02T04:00,12.3,84,0.0,1.4,0


### Step 5: Feature Selection and Train-Test Split

We select temperature, humidity, and wind speed as input features
and `rainy` as the target variable.

The dataset is divided into 80% training and 20% testing data.
Stratified splitting helps maintain the proportion of rainy
and non-rainy records in both sets.

In [6]:
from sklearn.model_selection import train_test_split

# Select input features
features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m"
]

X = clean_df[features]
y = clean_df["rainy"]

# Split into training and testing datasets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Display dataset sizes
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training samples: 153
Testing samples: 39

Training target distribution:
rainy
0    140
1     13
Name: count, dtype: int64

Testing target distribution:
rainy
0    36
1     3
Name: count, dtype: int64


### Step 6: KNN Model Training Using ML Pipeline

We create a machine learning pipeline with three steps:

1. **SimpleImputer:** Handles missing values using the median.
2. **StandardScaler:** Scales features so they contribute fairly to distance calculations.
3. **KNeighborsClassifier:** Classifies weather conditions using the 3 nearest neighbors.

The pipeline combines preprocessing and model training into one workflow.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Create KNN pipeline
knn_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=3))
])

# Train the model
knn_pipeline.fit(X_train, y_train)

# Make predictions
y_pred = knn_pipeline.predict(X_test)

print("KNN model trained successfully!")
print("Predictions:", y_pred)

KNN model trained successfully!
Predictions: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0
 0 0]
